In [ ]:
import shutil

shutil.unpack_archive('archive.zip', '/home/carl/Desktop/IDS_dev_v2/aegis-vanguard-network-attack-detection-system/detection_layers')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dataset1 = pd.read_csv('Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', encoding='latin-1')
dataset2 = pd.read_csv('Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', encoding='latin-1')
dataset3 = pd.read_csv('Friday-WorkingHours-Morning.pcap_ISCX.csv', encoding='latin-1')
dataset4 = pd.read_csv('Monday-WorkingHours.pcap_ISCX.csv', encoding='latin-1')
dataset5 = pd.read_csv('Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', encoding='latin-1')
dataset6 = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', encoding='latin-1')
dataset7 = pd.read_csv('Tuesday-WorkingHours.pcap_ISCX.csv', encoding='latin-1')
dataset8 = pd.read_csv('Wednesday-workingHours.pcap_ISCX.csv', encoding='latin-1')


In [ ]:
dataset_compiled = [dataset1, dataset2, dataset3, dataset4, dataset5, dataset6, dataset7, dataset8]

print('Dimensions of each dataset')
for idx, df in enumerate(dataset_compiled, start=1):
    row, col = df.shape
    print(f'Shape of dataset{idx}: rows = {row}, columns = {col}')

In [ ]:
# Check if all datasets have the same columns in preparation for compilation
for i in range(0, len(dataset_compiled) - 1):
    if dataset_compiled[i].columns.tolist() == dataset_compiled[i + 1].columns.tolist():
        continue
    else:
        raise ValueError("There's a column mismatch between the datasets")

In [ ]:
datasets_unified = pd.concat(dataset_compiled)
row, col = datasets_unified.shape

print(f"Specifications of the unified dataset: rows = {row}, columns = {col}")

In [ ]:
# datasets_unified.to_csv('compiled_datasets.csv')

In [ ]:
df = pd.read_csv('compiled_datasets.csv')
df.columns

In [ ]:
col_names = {col: col.strip() for col in df}
df = df.rename(columns=col_names)

In [ ]:
# Set the column view to None to prevent truncation
pd.options.display.max_columns = None
# Prevent long text inside a column from truncating
pd.options.display.max_colwidth = None

print('Overview of columns and features:')
df.head().transpose()

In [ ]:
df.info()

In [ ]:
df.describe().transpose()

In [ ]:
has_missing_val = df.count() != len(df)
print(f"Column with missing values: {has_missing_val[has_missing_val].index.tolist()}")

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number])

means = numeric_cols.mean()
stds = numeric_cols.std()
min_val = numeric_cols.min()
max_val = numeric_cols.max()

col_faulty_mean =  (means <= 0) | means.isin([np.inf, -np.inf, np.nan])
col_faulty_std  = (stds <= 0) | stds.isin([np.inf, -np.inf])
col_inf_min   = (min_val <= 0) | min_val.isin([np.inf, -np.inf])
col_inf_max   = (max_val <= 0) | max_val.isin([np.inf, -np.inf])

problematic_columns = {
    "zero_or_inf_mean": col_faulty_mean[col_faulty_mean].index.tolist(),
    "zero_or_inf_std": col_faulty_std[col_faulty_std].index.tolist(),
    "inf_min": col_inf_min[col_inf_min].index.tolist(),
    "inf_max": col_inf_max[col_inf_max].index.tolist()
}

print("Columns with mean <= 0 or infinite:", problematic_columns["zero_or_inf_mean"])
print("Columns with std <= 0 or infinite:", problematic_columns["zero_or_inf_std"])
print("Columns with min <= 0 or infinite:", problematic_columns["inf_min"])
print("Columns with max <= 0 or infinite:", problematic_columns["inf_max"])

1. The first column is just index numbers, so it should be dropped.
2. Flow Bytes/s and Flow Packets/s contains inf and NaN values
3. The mean, std, min, and max of the following columns are all zero:
Bwd PSH Flags
Bwd URG Flags
Fwd Avg Bytes/Bulk
Fwd Avg Packets/Bulk
Fwd Avg Bulk Rate
Bwd Avg Bytes/Bulk
Bwd Avg Packets/Bulk	
Bwd Avg Bulk Rate
4. Fwd Header Length.1 and Fwd Header Length are duplicates
5. Flow Bytes/s only have 2829385.0 counts against 2830743 total

## Data Cleaning